In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catlog_name}.{bronze_schema}.results"
silver_table = f"{catlog_name}.{silver_schema}.results"

In [0]:
results_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
results_selected_df = (
    results_df
    .select(
        F.col("date"),
        F.col("raceName"),
        F.col("season"),
        F.col("round"),
        F.col("constructorId"),
        F.col("driverId"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("status"),
        F.col("ingestion_timestamp"),
        F.col("source_file")
        
    )
)

In [0]:
display(results_selected_df)

In [0]:
results_renamed_df = (
    results_selected_df
    .withColumnsRenamed(
        {
            "date": "race_date",
            "raceName": "race_name",
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "positionText": "finish_position_text",
            "grid": "grid_position",
            "lap": "completed_lap",
            "number": "car_number",
            "position": "finish_position"
        }
    )
)

In [0]:
results_valid_df = (
    results_renamed_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("driver_id").isNotNull() &
        F.col("constructor_id").isNotNull()

    )
)

In [0]:
display(results_renamed_df.count() - results_valid_df.count())

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
result_final_df = (
    results_distinct_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
(
    result_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.read.table(silver_table))